In [ ]:
%load_ext autoreload
%autoreload 2


# KMeans-Based Model Example

This notebook shows how to run the packaged methylseg pathway using KMeans-derived state assignments before HMM segmentation.


In [ ]:
from pathlib import Path
import pandas as pd

from methylseg import (
    MethylDataPrep,
    MethylSegPathway,
    MethylStateAssignmentMethod,
    MethylationStates,
)
from methylseg.helper_classes import DATA_DIR

In [ ]:
REFERENCE_DIR = DATA_DIR / "reference_files"

OUT_DIR = "out" / "kmeans_based_model_output"


In [ ]:
model = MethylSegPathway.get_pretrained_model(OUT_DIR, resolution="450k")
model.state_assignment_method = MethylStateAssignmentMethod.KMEANS
model.segmentor.state_assignment_method = MethylStateAssignmentMethod.KMEANS


In [ ]:
sample_info, sample_info_removed = MethylDataPrep(
    meth_file=REFERENCE_DIR / "TCGA-BD-A3EP-01A_450k.tsv.gz",
    sample_id="TCGA-BD-A3EP-01A",
    resolution="450k",
    remove_low_coverage_like_cpgs=True,
).prepare()

sample_info.sample_id


In [ ]:
regions_chr1 = model.generate_regions(sample_info=sample_info, chrom="chr1")
regions_chr1.head()


In [ ]:
fig = model.plot_labels(
    label_source="kmeans",
    sample_info=sample_info,
    sample_info_removed=sample_info_removed,
    chrom="chr1",
    label_title="KMeans state",
)


In [ ]:
fig = model.plot_labels(
    label_source="hmm",
    sample_info=sample_info,
    sample_info_removed=sample_info_removed,
    chrom="chr1",
    label_title="HMM state",
)


In [ ]:
clean_summary_paths, clean_dir = model.get_clean_regions(
    regions_df=regions_chr1,
    sample_id=sample_info.sample_id,
    chrom="chr1",
)
clean_summary_paths, clean_dir


In [ ]:
fig = model.plot_labels(
    label_source="hmm",
    sample_info=sample_info,
    chrom="chr1",
    use_cleaned_regions=True,
    overlay_state="PMD",
    region_start=2_000_000,
    region_end=4_000_000,
    region_chrom="chr1",
    label_title="HMM state",
)


In [ ]:
region_paths = model.run_pathway(
    sample_info=sample_info,
    chroms=["chr1"],
    clean_regions=True,
)
region_paths


In [ ]:
pd.read_csv(region_paths[1], sep="\t", header=None).head()
